# Unravel Tax - Build Workbook Notebook

Milestone 2 notebook. It parses lightweight fixtures, calculates capital-gains buckets, and generates notebook outputs.

- M2A: structure and no-edit execution.
- M2B: parse CSV, Excel, HTML, and structured text into one normalized row shape.
- M2C: generate CA summary and full workbook outputs from notebook calculations.

PDF and free-form text remain routed through the guided prompt workflow, not through a custom parser here.

## 1. Setup

Run this first. In Colab, upload or clone the repo first; locally, run from the repo root.

In [ ]:
from pathlib import Path
import csv
import json
import sys

cwd = Path.cwd()
if (cwd / 'fixtures').exists():
    repo_root = cwd
elif (cwd.parent / 'fixtures').exists():
    repo_root = cwd.parent
else:
    raise RuntimeError('Run this notebook from the Unravel Tax repo root, or from notebooks/.')

fixtures_dir = repo_root / 'fixtures'
template_path = repo_root / 'templates' / 'excel-export' / 'UnravelTax-Template.xlsx'
m1c_dir = repo_root / 'dry-runs' / 'm1c'
output_dir = repo_root / 'notebooks' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)

assert template_path.exists(), template_path
scripts_dir = repo_root / 'scripts'
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

from notebook_engine import load_transactions, summarize_transactions, validate_fixture_parity
from notebook_engine import validate_ca_summary_against_reference, write_ca_summary_csv, write_full_workbook

print(f'Repo root: {repo_root}')
print(f'Output dir: {output_dir}')

## 2. Choose A Fixture

The notebook uses synthetic fixtures only. CSV, Excel, HTML, and structured text are parsed deterministically; PDF/free-form text is routed to the guided prompt workflow.

In [ ]:
fixture_options = {
    'csv': fixtures_dir / 'sample-broker-statement.csv',
    'excel': fixtures_dir / 'sample-broker-statement.xlsx',
    'html': fixtures_dir / 'sample-broker-statement.html',
    'structured_text': fixtures_dir / 'sample-broker-statement.tsv',
    'pdf_extracted_text': fixtures_dir / 'sample-pdf-extracted-text.txt',
}

selected_fixture_kind = 'html'
selected_fixture = fixture_options[selected_fixture_kind]
assert selected_fixture.exists(), selected_fixture

print(f'Selected fixture: {selected_fixture_kind} -> {selected_fixture.name}')

## 3. Ingestion And Calculations

Parse every lightweight fixture into the same normalized transaction shape, calculate holding period, tax class, and gain/loss, and validate parity against the CSV baseline.

In [ ]:
parity_state = validate_fixture_parity(fixture_options)
selected_transactions = load_transactions(selected_fixture_kind, selected_fixture)
assert isinstance(selected_transactions, list)
selected_summary = summarize_transactions(selected_transactions)

reference_ca_summary = m1c_dir / 'UnravelTax-M1C-CA-Summary.csv'
assert reference_ca_summary.exists(), reference_ca_summary
with reference_ca_summary.open(newline='', encoding='utf-8') as handle:
    ca_summary_rows = list(csv.DictReader(handle))
reference_amounts = {row['Head']: row['Amount'] for row in ca_summary_rows}
reference_m1_summary = {
    'intraday_gain': float(reference_amounts['Speculative / Intraday income']),
    'stcg': float(reference_amounts['Short-Term Capital Gains']),
    'ltcg': float(reference_amounts['Long-Term Capital Gains']),
}
assert selected_summary['intraday_gain'] == reference_m1_summary['intraday_gain']
assert selected_summary['stcg'] == reference_m1_summary['stcg']
assert selected_summary['ltcg'] == reference_m1_summary['ltcg']

calculation_state = {
    'milestone': 'M2C notebook exports',
    'selected_fixture_kind': selected_fixture_kind,
    'selected_fixture_name': selected_fixture.name,
    'parsed_fixture_kinds': parity_state['parsed_fixture_kinds'],
    'pdf_extracted_text_route': parity_state['pdf_extracted_text_route'],
    'selected_summary': selected_summary,
    'reference_m1_summary': reference_m1_summary,
    'reference_ca_summary_rows': len(ca_summary_rows),
    'next_step': 'M4D adds the guided UI and reconciliation panel after calculation parity.',
}

print(json.dumps(calculation_state, indent=2))

## 4. Generate Exports

Generate the two notebook outputs from parsed transactions: a CA Summary CSV and a full workbook. The CA Summary is validated against the Milestone 1 reference totals.

In [ ]:
notebook_ca_summary = output_dir / 'UnravelTax-Notebook-CA-Summary.csv'
notebook_full_workbook = output_dir / 'UnravelTax-Notebook-Full-Workbook.xlsx'
manifest_path = output_dir / 'm2c-notebook-exports-manifest.json'

write_ca_summary_csv(selected_transactions, notebook_ca_summary)
write_full_workbook(selected_transactions, notebook_full_workbook)
validate_ca_summary_against_reference(notebook_ca_summary, reference_ca_summary)

manifest = {
    **calculation_state,
    'outputs': [
        str(notebook_ca_summary.relative_to(repo_root)),
        str(notebook_full_workbook.relative_to(repo_root)),
    ],
    'note': 'M2C generated notebook outputs and validated CA Summary totals against M1C.',
}
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')

print(f'Wrote {notebook_ca_summary}')
print(f'Wrote {notebook_full_workbook}')
print(f'Wrote {manifest_path}')

## 5. Readiness Check

If this cell passes, Milestone 2 can proceed to rules-as-data and reconciliation.

In [ ]:
assert notebook_ca_summary.exists()
assert notebook_full_workbook.exists()
assert manifest_path.exists()
assert calculation_state['selected_summary']['rows'] == 5
assert calculation_state['selected_summary']['intraday_gain'] == 800
assert calculation_state['selected_summary']['stcg'] == -500
assert calculation_state['selected_summary']['ltcg'] == 5500
print('M2C notebook exports ran end to end without manual edits.')